# ITO5202 Assessment 2: Machine Learning and Streaming Analytics

**Student ID:** 27602966  
**Dataset:** NYC TLC Yellow Taxi Trip Records, January–December 2023, and Taxi Zone Lookup  
**GitHub repository:** [ITO5202_Ass2_27602966](https://github.com/hncha9-eng/ITO5202_Ass2_27602966)

## Assessment Overview

This assessment extends the historical batch analysis completed in Assessment 1. The approved dataset is reused to train and evaluate a Spark ML model, then apply the persisted model to previously unseen records replayed through Kafka.

| Notebook section | Requirements addressed |
|---|---|
| Execution Environment | Runtime configuration, dependency versions and environment verification |
| Dataset Reuse and Train/Stream Split | Source validation, data preparation, reproducible partitioning and streaming subset persistence |
| Part A: The ML Pipeline | Task selection, feature engineering, model comparison, evaluation, model persistence and reload verification |
| Part B: Streaming, Kafka and Reflection | Held-out replay, Kafka producer, Structured Streaming consumer, windowed aggregation and embedded reflection |

The streaming subset will remain excluded from model fitting, tuning and selection. Part B will use the persisted feature engineering and prediction pipeline from Part A.

## Development and Commit Plan

Development is organised into 13 commit milestones. Commits are made as the corresponding work is completed and verified. Additional debugging or refinement commits may be included where needed.

The existing environment and taxi lookup commits establish the project foundation. The four required assessment phases are represented explicitly below.

| No. | Phase | Development milestone | Commit message |
|---|---|---|---|
| 1 | Foundation | Environment setup and initial reproduction documentation | Set up Spark and Kafka environment with reproducible documentation |
| 2 | Foundation | Add the taxi zone reference data | Add NYC taxi zone lookup data |
| 3 | Foundation | Organise the notebook around assessment requirements | Align assessment notebook with assessment requirements |
| 4 | Phase 1 | Standardise monthly schemas and document data quality checks | Standardise taxi schemas and validate data quality |
| 5 | Phase 1 | Implement the seeded train/stream split and verify saved streaming data | Implement reproducible split and persist streaming holdout |
| 6 | Phase 1 | Explore training data, justify features and construct the preprocessing pipeline | Explore training data and implement feature engineering |
| 7 | Phase 2 | Train competing models and present their evaluation results | Train and compare competing Spark ML models |
| 8 | Phase 2 | Justify the final model, persist its complete pipeline and verify reloaded predictions | Select and persist the verified best pipeline |
| 9 | Phase 3 | Implement the standalone producer with parameterised batches and publication timestamps | Implement held-out Kafka batch producer |
| 10 | Phase 3 | Test producer execution, retain batch logs and document exact Kafka commands | Verify producer timing and document Kafka execution |
| 11 | Phase 4 | Implement streaming ingestion, explicit JSON parsing and persisted-model inference with prediction sinks | Integrate Kafka streaming with persisted model inference |
| 12 | Phase 4 | Implement sliding windows, watermarking and visible aggregation results | Add watermarked sliding-window prediction monitoring |
| 13 | Phase 4 and submission | Complete the evidence-based reflection, reproduction documentation and notebook/PDF checks | Complete reflection and verify submission documentation |

### Required Phase Coverage

- **Phase 1 — Commits 4–6:** train/stream split, data exploration and feature engineering.
- **Phase 2 — Commits 7–8:** model training, comparison, evaluation and persistence.
- **Phase 3 — Commits 9–10:** Kafka producer development and local testing.
- **Phase 4 — Commits 11–13:** Structured Streaming consumer, windowed aggregation and reflection.

Commit identifiers will be recorded after the commits exist. Notebook outputs and written interpretations will provide evidence for the completed milestones.

## Execution Environment

### Python and Spark Configuration

The notebook runs inside the PySpark Docker container, with the repository mounted at `/home/student/work`. Project files are accessed using paths relative to the repository root.

The configuration below matches the verified assessment environment.

| Setting | Configuration |
|---|---|
| Apache Spark | 3.5.5 |
| Scala | 2.12.18 |
| Execution mode | local[2] |
| Driver memory | 2 GB |
| Spark session timezone | America/New_York |
| Kafka connector | org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5 |
| Docker network | ito5202 |
| Kafka bootstrap server inside the Docker network | kafka:9092 |

The Kafka connector matches the Spark version and Scala binary version. Package and driver memory settings are declared before Spark starts.

The `America/New_York` session timezone supports consistent interpretation of the NYC taxi timestamps. The same Spark environment will support historical model development in Part A and streaming inference in Part B.

In [1]:
import os
import sys
from pathlib import Path

import pyspark

EXPECTED_SPARK_VERSION = "3.5.5"

if pyspark.__version__ != EXPECTED_SPARK_VERSION:
    raise RuntimeError(
        f"Expected PySpark {EXPECTED_SPARK_VERSION}, "
        f"found {pyspark.__version__}. "
        "Use the assessment's verified container."
    )

KAFKA_CONNECTOR = (
    "org.apache.spark:"
    "spark-sql-kafka-0-10_2.12:3.5.5"
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--driver-memory 2g --packages {KAFKA_CONNECTOR} pyspark-shell"
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 2")
    .master("local[2]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

### Environment Verification

The following cell records the runtime versions and active Spark configuration so the execution environment can be checked during reproduction.

A three-row DataFrame is displayed to verify that a Spark action completes successfully. The expected configuration is Spark 3.5.5, Scala 2.12.18, `local[2]`, driver memory `2g` and session timezone `America/New_York`.

In [2]:
print("Python:", sys.version.split()[0])
print("Spark:", spark.version)
print(
    "Scala:",
    spark.sparkContext._jvm.scala.util.Properties.versionNumberString()
)
print("Master:", spark.sparkContext.master)
print(
    "Driver memory:",
    spark.sparkContext.getConf().get("spark.driver.memory")
)
print("Timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Kafka connector:", KAFKA_CONNECTOR)

spark.range(3).show()

Python: 3.10.19
Spark: 3.5.5
Scala: 2.12.18
Master: local[2]
Driver memory: 2g
Timezone: America/New_York
Kafka connector: org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



# Dataset Reuse and Train/Stream Split

## 1. Dataset Reuse and Preparation

### 1.1 Dataset Sources and Input Verification

Assessment 2 reuses the approved Assessment 1 dataset: NYC TLC Yellow Taxi Trip Records for January–December 2023 and the Taxi Zone Lookup. No new dataset selection is required.

| Source | Expected files | Purpose |
|---|---|---|
| Yellow Taxi Trip Records | Twelve monthly Parquet files, from `yellow_tripdata_2023-01.parquet` to `yellow_tripdata_2023-12.parquet` | Trip records containing timestamps, location identifiers, passenger counts, distances and financial attributes |
| Taxi Zone Lookup | `taxi_zone_lookup.csv` | Reference attributes associated with taxi location identifiers |

All source files are stored in the repository's `data/` directory. 

The following cell verifies that all thirteen required source files are available. Schema inspection and data validation follow in subsequent sections.

The large Parquet files are excluded from Git tracking. Their download instructions are documented in `data/README.md`, while the small taxi zone lookup is included in the repository.

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"

MONTHLY_FILES = [
    DATA_DIR / f"yellow_tripdata_2023-{month:02d}.parquet"
    for month in range(1, 13)
]

ZONE_FILE = DATA_DIR / "taxi_zone_lookup.csv"
required_files = MONTHLY_FILES + [ZONE_FILE]

print("Repository root:", PROJECT_ROOT)

missing_files = []

for source_file in required_files:
    exists = source_file.is_file()

    print(
        f"{'FOUND' if exists else 'MISSING'}: {source_file.name}"
    )

    if not exists:
        missing_files.append(source_file.name)

if missing_files:
    raise FileNotFoundError(
        "Place the original source files in the repository's data folder. "
        f"Missing: {', '.join(missing_files)}"
    )

Repository root: /home/student/work
FOUND: yellow_tripdata_2023-01.parquet
FOUND: yellow_tripdata_2023-02.parquet
FOUND: yellow_tripdata_2023-03.parquet
FOUND: yellow_tripdata_2023-04.parquet
FOUND: yellow_tripdata_2023-05.parquet
FOUND: yellow_tripdata_2023-06.parquet
FOUND: yellow_tripdata_2023-07.parquet
FOUND: yellow_tripdata_2023-08.parquet
FOUND: yellow_tripdata_2023-09.parquet
FOUND: yellow_tripdata_2023-10.parquet
FOUND: yellow_tripdata_2023-11.parquet
FOUND: yellow_tripdata_2023-12.parquet
FOUND: taxi_zone_lookup.csv


#### Input Verification Results

All twelve monthly Parquet files and `taxi_zone_lookup.csv` were reported as `FOUND`. The expected source files are therefore available for schema inspection and loading.

These inputs preserve the dataset coverage used in Assessment 1. The subsequent preparation steps will establish a consistent schema and document data quality decisions before the reproducible train/stream split.

### 1.2 Schema Definition and Source Inspection

The twelve monthly files are inspected separately to identify differences in column names and data types before combining their records.

Assessment 1 identified schema differences involving `passenger_count`, `RatecodeID`, pickup and drop-off location identifiers, and the capitalisation of the airport fee column.

The following comparison documents these differences using each file's Parquet schema. The findings guide the common schema used in the subsequent loading and standardisation step.

Statistical exploration and learned feature preprocessing will be performed using the training data after the train/stream split.

In [4]:
monthly_schemas = {}
schema_rows = []

for source_file in MONTHLY_FILES:
    source_df = spark.read.parquet(str(source_file))
    source_schema = source_df.schema

    monthly_schemas[source_file.name] = source_schema

    column_types = {
        field.name: field.dataType.simpleString()
        for field in source_schema.fields
    }

    airport_columns = [
        field.name
        for field in source_schema.fields
        if field.name.lower() == "airport_fee"
    ]

    schema_rows.append((
        source_file.stem[-7:],
        column_types.get("passenger_count", "missing"),
        column_types.get("RatecodeID", "missing"),
        column_types.get("PULocationID", "missing"),
        column_types.get("DOLocationID", "missing"),
        ", ".join(airport_columns) or "missing",
    ))

schema_summary = spark.createDataFrame(
    schema_rows,
    schema=(
        "month string, passenger_count string, RatecodeID string, "
        "PULocationID string, DOLocationID string, airport_column string"
    ),
)

schema_summary.orderBy("month").show(12, truncate=False)

+-------+---------------+----------+------------+------------+--------------+
|month  |passenger_count|RatecodeID|PULocationID|DOLocationID|airport_column|
+-------+---------------+----------+------------+------------+--------------+
|2023-01|double         |double    |bigint      |bigint      |airport_fee   |
|2023-02|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-03|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-04|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-05|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-06|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-07|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-08|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-09|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-10|bigint         |bigint    |int         |int         |Ai

#### January Source Schema

The complete January schema provides a reference for the original nineteen columns, including trip timestamps, numerical attributes, location identifiers and categorical fields.

Displaying the full schema makes the source types visible before defining the common schema used across all months.

In [5]:
january_df = spark.read.parquet(str(MONTHLY_FILES[0]))

print("Source:", MONTHLY_FILES[0].name)
january_df.printSchema()

Source: yellow_tripdata_2023-01.parquet
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



#### Schema Inspection Findings

The executed monthly comparison confirms the following differences:

| Field | January 2023 | February–December 2023 |
|---|---|---|
| `passenger_count` | `double` | `bigint` |
| `RatecodeID` | `double` | `bigint` |
| `PULocationID` | `bigint` | `int` |
| `DOLocationID` | `bigint` | `int` |
| Airport fee column name | `airport_fee` | `Airport_fee` |

The January file contains nineteen columns. Both `tpep_pickup_datetime` and `tpep_dropoff_datetime` use `timestamp_ntz`. Distance and financial attributes use `double`, while `store_and_fwd_flag` uses `string`.

These findings establish the need for a consistent loading strategy. The next section will read each monthly file separately, apply compatible column types, standardise the airport fee column name and combine the resulting DataFrames.

### 1.3 Monthly Data Loading and Standardisation

Each monthly file is read using its source Parquet schema, then transformed into a common nineteen-column schema before the months are combined.

This approach handles the differences identified in Section 1.2 through explicit casts and consistent column naming.

| Columns | Common type or name | Rationale |
|---|---|---|
| `VendorID`, `PULocationID`, `DOLocationID`, `payment_type` | `long` | Provides consistent integer storage across the source files |
| `passenger_count`, `RatecodeID` | `double` | Retains fractional values from the January source for explicit data quality checks |
| Pickup and drop-off datetimes | `timestamp` | Provides a common timestamp type under the configured `America/New_York` session timezone |
| `trip_distance` and financial attributes | `double` | Provides consistent numerical types across all months |
| `store_and_fwd_flag` | `string` | Retains the original categorical representation |
| Airport fee column | `airport_fee` | Standardises the capitalisation difference between January and subsequent months |

Keeping `passenger_count` and `RatecodeID` as doubles during preparation allows questionable fractional values to be identified before deciding how to handle them.

The loader checks that each file contains all required columns. It then selects, casts and aliases those columns in a consistent order.

The monthly DataFrames are combined using `unionByName()`, which aligns corresponding columns by name. All source rows are retained at this stage; eligibility rules will be documented and applied during data quality preparation.

These storage types establish a consistent source schema. Feature selection and the treatment of numerical and categorical inputs will be justified in Part A.

In [6]:
from functools import reduce
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/New_York")

COLUMN_TYPES = {
    "VendorID": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "store_and_fwd_flag": "string",
    "PULocationID": "long",
    "DOLocationID": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
}


def standardise_month(source_path):
    source_df = spark.read.parquet(str(source_path))

    source_columns = {
        name.lower(): name
        for name in source_df.columns
    }

    missing_columns = [
        name
        for name in COLUMN_TYPES
        if name.lower() not in source_columns
    ]

    if missing_columns:
        raise ValueError(
            f"{source_path.name}: missing columns {missing_columns}"
        )

    return source_df.select(*[
        F.col(source_columns[name.lower()])
        .cast(data_type)
        .alias(name)
        for name, data_type in COLUMN_TYPES.items()
    ])


monthly_dfs = [
    standardise_month(source_path)
    for source_path in MONTHLY_FILES
]

raw_trips_df = reduce(
    lambda combined, month: combined.unionByName(month),
    monthly_dfs,
)

raw_trips_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



#### Standardisation Outcome

The combined DataFrame, `raw_trips_df`, uses the common nineteen-column schema defined above.

The airport fee column has the consistent name `airport_fee`. Location identifiers use `long`, while passenger counts and rate codes remain `double` for subsequent validation.

This DataFrame provides the source for record-count verification and data quality preparation. The next section checks the loaded dataset's size and displays a bounded sample.

### 1.4 Source Loading Validation

The combined DataFrame is checked for the number of monthly inputs, column count and total record count.

The record count is compared with the original raw dataset count from Assessment 1 to verify consistency in dataset coverage. A five-row sample displays selected fields after standardisation.

These results establish the baseline against which subsequent data quality exclusions and train/stream partition counts will be reconciled.

In [7]:
raw_trip_count = raw_trips_df.count()

print("Monthly files:", len(monthly_dfs))
print("Columns:", len(raw_trips_df.columns))
print(f"Total trip records: {raw_trip_count:,}")

raw_trips_df.select(
    "tpep_pickup_datetime",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "airport_fee",
).show(5, truncate=False)

Monthly files: 12
Columns: 19
Total trip records: 38,310,226
+--------------------+---------------+----------+------------+-----------+
|tpep_pickup_datetime|passenger_count|RatecodeID|PULocationID|airport_fee|
+--------------------+---------------+----------+------------+-----------+
|2023-01-01 00:32:10 |1.0            |1.0       |161         |0.0        |
|2023-01-01 00:55:08 |1.0            |1.0       |43          |0.0        |
|2023-01-01 00:25:04 |1.0            |1.0       |48          |0.0        |
|2023-01-01 00:03:48 |0.0            |1.0       |138         |1.25       |
|2023-01-01 00:10:29 |1.0            |1.0       |107         |0.0        |
+--------------------+---------------+----------+------------+-----------+
only showing top 5 rows



#### Source Loading Results

The executed validation produced the following results:

| Check | Observed result |
|---|---:|
| Monthly source files | 12 |
| Combined columns | 19 |
| Total raw trip records | 38,310,226 |

The combined record count matches the raw dataset count reported in Assessment 1. This establishes consistent dataset coverage before eligibility filtering.

The displayed sample shows the standardised passenger count, rate code, pickup location and airport fee fields alongside the pickup timestamp. It also contains a passenger count of `0.0`, whose treatment will be documented during data quality preparation.

The raw count of 38,310,226 will remain the reference for reporting retained records, excluded records and the subsequent train/stream split. Data quality checks will determine the eligible population used for that split.

### 1.5 Data Quality Checks and Eligibility Rules

#### Eligibility Rules

The unit of analysis is an individual trip record. Eligibility follows the scope used in Assessment 1, with explicit finite-value checks added for numerical fields.

| Requirement | Eligibility rule | Rationale |
|---|---|---|
| Pickup date | Falls within January–December 2023 | Preserves the approved period of analysis |
| Trip distance | Present, finite and greater than zero | Restricts the population to trips with positive reported travel distance |
| Trip duration | Present, finite and greater than zero | Requires a recorded drop-off later than pickup |
| Fare amount | Present, finite and non-negative | Preserves Assessment 1's fare eligibility rule |

Trip duration is calculated in minutes from the difference between drop-off and pickup timestamps.

Missing, non-finite, non-positive or fractional passenger counts are reported as diagnostic issues. Missing or fractional rate codes and missing location identifiers are also reported. These diagnostics do not introduce additional row-exclusion rules at this stage. Their treatment will be documented during feature preparation.

These eligibility rules are fixed independently of model performance. Learned imputation, scaling and distribution-based outlier decisions will use the fitting data after the train/stream split.

In [8]:
from functools import reduce
from pyspark.sql import functions as F

quality_source_df = (
    raw_trips_df
    .withColumn(
        "pickup_date",
        F.to_date("tpep_pickup_datetime"),
    )
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0,
    )
)


def is_finite_numeric(column_name):
    value = F.col(column_name)

    return F.coalesce(
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != F.lit(float("inf"))),
        F.lit(False),
    )


pickup_in_2023 = F.coalesce(
    F.col("pickup_date").between("2023-01-01", "2023-12-31"),
    F.lit(False),
)

validity_checks = {
    "pickup_in_2023": pickup_in_2023,
    "positive_distance": (
        is_finite_numeric("trip_distance")
        & (F.col("trip_distance") > 0)
    ),
    "positive_duration": (
        is_finite_numeric("trip_duration_minutes")
        & (F.col("trip_duration_minutes") > 0)
    ),
    "non_negative_fare": (
        is_finite_numeric("fare_amount")
        & (F.col("fare_amount") >= 0)
    ),
}

eligible_condition = reduce(
    lambda left, right: left & right,
    validity_checks.values(),
)

# Original Assessment 1 rules, for comparison on the current data.
a1_condition = F.coalesce(
    pickup_in_2023
    & (F.col("trip_distance") > 0)
    & (F.col("trip_duration_minutes") > 0)
    & (F.col("fare_amount") >= 0),
    F.lit(False),
)

exclusion_checks = {
    f"fails_{name}": ~condition
    for name, condition in validity_checks.items()
}

diagnostic_checks = {
    "passenger_missing_nonfinite": (
        ~is_finite_numeric("passenger_count")
    ),
    "passenger_nonpositive": (
        is_finite_numeric("passenger_count")
        & (F.col("passenger_count") <= 0)
    ),
    "passenger_fractional": (
        is_finite_numeric("passenger_count")
        & (
            F.col("passenger_count")
            != F.floor("passenger_count")
        )
    ),
    "ratecode_missing_nonfinite": (
        ~is_finite_numeric("RatecodeID")
    ),
    "ratecode_fractional": (
        is_finite_numeric("RatecodeID")
        & (F.col("RatecodeID") != F.floor("RatecodeID"))
    ),
    "pickup_location_missing": F.col("PULocationID").isNull(),
    "dropoff_location_missing": F.col("DOLocationID").isNull(),
}

quality_checks = {
    **exclusion_checks,
    **diagnostic_checks,
}

eligible_trips_df = quality_source_df.filter(eligible_condition)

#### Quality Audit and Reconciliation

The following aggregation calculates the raw, eligible and excluded record counts, together with the count and percentage affected by each quality check.

Individual issue counts can overlap because one record may fail several checks. The excluded-record total therefore counts records failing at least one eligibility requirement.

Checks labelled `Report only` provide diagnostic evidence without adding exclusion criteria. Their percentages use the full raw dataset as the denominator.

The current data is also evaluated under the original Assessment 1 rules. This separates any difference from the previous eligible count from additional exclusions introduced by the finite-value checks.

All counts are calculated together in one aggregation over the source DataFrame. Only the resulting summary row is returned to Python.

In [9]:
from IPython.display import Markdown, display

quality_counts = quality_source_df.agg(
    F.count("*").alias("raw_records"),
    F.sum(
        F.when(a1_condition, 1).otherwise(0)
    ).alias("a1_rule_records"),
    F.sum(
        F.when(eligible_condition, 1).otherwise(0)
    ).alias("eligible_records"),
    F.sum(
        F.when(~eligible_condition, 1).otherwise(0)
    ).alias("excluded_records"),
    *[
        F.sum(
            F.when(condition, 1).otherwise(0)
        ).alias(name)
        for name, condition in quality_checks.items()
    ],
).first().asDict()

total_checked = int(quality_counts["raw_records"])
eligible_trip_count = int(quality_counts["eligible_records"])
excluded_trip_count = int(quality_counts["excluded_records"])
a1_rule_count = int(quality_counts["a1_rule_records"])

assert total_checked == raw_trip_count, (
    "Source count changed; check the input files."
)
assert total_checked == eligible_trip_count + excluded_trip_count, (
    "Eligible and excluded counts do not reconcile."
)
assert eligible_trip_count > 0, "No eligible records remain."

summary_rows = [
    (
        stage,
        count,
        round(100.0 * count / total_checked, 4),
    )
    for stage, count in [
        ("Raw records", total_checked),
        ("Eligible records", eligible_trip_count),
        ("Excluded records", excluded_trip_count),
    ]
]

spark.createDataFrame(
    summary_rows,
    "stage string, record_count long, pct_raw double",
).show(truncate=False)

issue_rows = [
    (
        name,
        int(quality_counts[name]),
        round(100.0 * quality_counts[name] / total_checked, 4),
        "Exclude" if name in exclusion_checks else "Report only",
    )
    for name in quality_checks
]

spark.createDataFrame(
    issue_rows,
    "check string, record_count long, pct_raw double, action string",
).show(len(issue_rows), truncate=False)

additional_numeric_exclusions = a1_rule_count - eligible_trip_count

print(f"Assessment 1 reference eligible records: {37_192_160:,}")
print(f"Current records passing Assessment 1 rules: {a1_rule_count:,}")
print(
    "Difference from Assessment 1 reference:",
    f"{a1_rule_count - 37_192_160:+,}",
)
print(
    "Additional finite-value exclusions:",
    f"{additional_numeric_exclusions:,}",
)

+----------------+------------+-------+
|stage           |record_count|pct_raw|
+----------------+------------+-------+
|Raw records     |38310226    |100.0  |
|Eligible records|37192160    |97.0815|
|Excluded records|1118066     |2.9185 |
+----------------+------------+-------+

+---------------------------+------------+-------+-----------+
|check                      |record_count|pct_raw|action     |
+---------------------------+------------+-------+-----------+
|fails_pickup_in_2023       |104         |3.0E-4 |Exclude    |
|fails_positive_distance    |773457      |2.0189 |Exclude    |
|fails_positive_duration    |15569       |0.0406 |Exclude    |
|fails_non_negative_fare    |381650      |0.9962 |Exclude    |
|passenger_missing_nonfinite|1309356     |3.4178 |Report only|
|passenger_nonpositive      |583005      |1.5218 |Report only|
|passenger_fractional       |0           |0.0    |Report only|
|ratecode_missing_nonfinite |1309356     |3.4178 |Report only|
|ratecode_fractional      

#### Data Quality Findings and Implications

The eligibility checks retain 37,192,160 records (97.08%) and exclude 1,118,066 records (2.92%). The retained count matches Assessment 1, and the explicit finite-value checks introduce no additional exclusions.

The distance requirement has the largest failure count: 773,457 records (2.0189%). The fare requirement fails for 381,650 records (0.9962%), while the duration requirement fails for 15,569 records (0.0406%). Another 104 records fail the pickup-date requirement. These counts overlap because individual records can fail multiple requirements.

Passenger counts are missing or non-finite in 1,309,356 records (3.4178%), and another 583,005 records (1.5218%) have finite, non-positive passenger counts. Rate codes are missing or non-finite in 1,309,356 records (3.4178%). These diagnostic counts describe the raw dataset; their prevalence within the fitting data will be examined after partitioning.

No finite fractional passenger counts or rate codes were detected. Both location identifier columns contain no missing values. Their coverage against the taxi zone lookup will be verified separately.

The passenger-count and rate-code findings establish the need for explicit feature-handling rules. Any learned imputation values will be fitted using the fitting partition and persisted within the pipeline for consistent application during streaming inference.

### 1.6 Taxi Zone Lookup and Join Validation

#### Lookup Schema and Key Validation

The taxi zone lookup supplies descriptive attributes for pickup and drop-off location identifiers.

| Column | Declared type | Purpose |
|---|---|---|
| `LocationID` | `long` | Reference key matching the trip location identifiers |
| `Borough` | `string` | Borough description |
| `Zone` | `string` | Taxi zone description |
| `service_zone` | `string` | Service zone classification |

The CSV is loaded with an explicit schema. Before joining, the lookup is checked for missing and duplicate location identifiers. Unique reference keys ensure that each trip can match at most one lookup row for each location.

Whitespace is trimmed from descriptive attributes. Missing or empty descriptions are represented as `Unknown`. These are deterministic reference-data preparation rules.

In [10]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
)

ZONE_SCHEMA = StructType([
    StructField("LocationID", LongType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

zone_df = (
    spark.read
    .option("header", "true")
    .option("enforceSchema", "false")
    .option("mode", "FAILFAST")
    .option("nullValue", "\\N")
    .schema(ZONE_SCHEMA)
    .csv(str(ZONE_FILE))
)

lookup_stats = zone_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("LocationID").alias("distinct_location_ids"),
    F.sum(
        F.when(F.col("LocationID").isNull(), 1).otherwise(0)
    ).alias("missing_location_ids"),
).first().asDict()

print("Lookup records:", lookup_stats["row_count"])
print("Distinct location IDs:", lookup_stats["distinct_location_ids"])
print("Missing location IDs:", lookup_stats["missing_location_ids"])

assert lookup_stats["row_count"] > 0, "The lookup is empty."
assert lookup_stats["missing_location_ids"] == 0, (
    "The lookup contains missing location identifiers."
)
assert (
    lookup_stats["row_count"]
    == lookup_stats["distinct_location_ids"]
), "Duplicate location identifiers would multiply joined trip records."

for attribute in ["Borough", "Zone", "service_zone"]:
    text_value = F.trim(F.col(attribute))

    zone_df = zone_df.withColumn(
        attribute,
        F.when(
            text_value.isNull() | (text_value == ""),
            F.lit("Unknown"),
        ).otherwise(text_value),
    )

zone_df.printSchema()
zone_df.orderBy("LocationID").show(5, truncate=False)

Lookup records: 265
Distinct location IDs: 265
Missing location IDs: 0
root
 |-- LocationID: long (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows



#### Pickup and Drop-off Enrichment

Two copies of the lookup are created with separate pickup and drop-off attribute names.

Left joins preserve eligible trip records when a location identifier has no reference match. The lookup tables are broadcast because they contain only a few hundred reference records compared with millions of trip records.

Explicit match flags distinguish an unmatched key from a matched key whose description is `Unknown`. Unmatched descriptions are filled with `Unknown`, while their match flags are set to `False`.

The enriched DataFrame will form the input to the train/stream split. Its reference attributes will also be included in the saved streaming subset, providing consistent input columns for batch modelling and streaming inference.

Feature inclusion, exclusion and categorical encoding decisions will be documented in Part A.

In [11]:
pickup_zone_df = zone_df.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
    F.col("service_zone").alias("pickup_service_zone"),
    F.lit(True).alias("pickup_lookup_matched"),
)

dropoff_zone_df = zone_df.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("service_zone").alias("dropoff_service_zone"),
    F.lit(True).alias("dropoff_lookup_matched"),
)

ZONE_ATTRIBUTE_COLUMNS = [
    "pickup_borough",
    "pickup_zone",
    "pickup_service_zone",
    "dropoff_borough",
    "dropoff_zone",
    "dropoff_service_zone",
]

LOOKUP_MATCH_COLUMNS = [
    "pickup_lookup_matched",
    "dropoff_lookup_matched",
]

prepared_trips_df = (
    eligible_trips_df
    .join(
        F.broadcast(pickup_zone_df),
        on="PULocationID",
        how="left",
    )
    .join(
        F.broadcast(dropoff_zone_df),
        on="DOLocationID",
        how="left",
    )
    .fillna("Unknown", subset=ZONE_ATTRIBUTE_COLUMNS)
    .fillna(False, subset=LOOKUP_MATCH_COLUMNS)
)

#### Join Cardinality and Reference Coverage

The following audit checks that the joined record count equals the eligible count established in Section 1.5.

Unmatched pickup and drop-off keys are counted separately using the explicit match flags. These counts measure reference-key coverage, including matches to reference rows with unknown descriptions.

The joined count and both coverage counts are calculated in one aggregation. A bounded sample displays the resulting pickup and drop-off attributes.

In [12]:
join_stats = prepared_trips_df.agg(
    F.count("*").alias("joined_records"),
    F.sum(
        F.when(~F.col("pickup_lookup_matched"), 1).otherwise(0)
    ).alias("unmatched_pickup_records"),
    F.sum(
        F.when(~F.col("dropoff_lookup_matched"), 1).otherwise(0)
    ).alias("unmatched_dropoff_records"),
).first().asDict()

joined_trip_count = int(join_stats["joined_records"])
unmatched_pickup_count = int(join_stats["unmatched_pickup_records"])
unmatched_dropoff_count = int(join_stats["unmatched_dropoff_records"])

join_audit_rows = [
    (
        check,
        count,
        round(100.0 * count / eligible_trip_count, 4),
    )
    for check, count in [
        ("Eligible records before joins", eligible_trip_count),
        ("Records after zone joins", joined_trip_count),
        ("Unmatched pickup keys", unmatched_pickup_count),
        ("Unmatched dropoff keys", unmatched_dropoff_count),
    ]
]

spark.createDataFrame(
    join_audit_rows,
    "check string, record_count long, pct_eligible double",
).show(truncate=False)

assert joined_trip_count == eligible_trip_count, (
    "The joins changed the eligible record count."
)

prepared_trips_df.select(
    F.col("PULocationID").alias("PU_ID"),
    F.col("pickup_borough").alias("PU_Borough"),
    F.col("pickup_zone").alias("PU_Zone"),
    F.col("DOLocationID").alias("DO_ID"),
    F.col("dropoff_borough").alias("DO_Borough"),
    F.col("dropoff_zone").alias("DO_Zone"),
).show(5, truncate=25)

display(Markdown(
    f"**Join validation:** The lookup contains "
    f"**{lookup_stats['row_count']:,}** records with unique, "
    f"non-missing location identifiers. Both joins preserve "
    f"all **{joined_trip_count:,}** eligible trip records. "
    f"There are **{unmatched_pickup_count:,}** unmatched pickup "
    f"keys and **{unmatched_dropoff_count:,}** unmatched "
    f"drop-off keys."
))

+-----------------------------+------------+------------+
|check                        |record_count|pct_eligible|
+-----------------------------+------------+------------+
|Eligible records before joins|37192160    |100.0       |
|Records after zone joins     |37192160    |100.0       |
|Unmatched pickup keys        |0           |0.0         |
|Unmatched dropoff keys       |0           |0.0         |
+-----------------------------+------------+------------+

+-----+----------+-----------------+-----+----------+---------------------+
|PU_ID|PU_Borough|          PU_Zone|DO_ID|DO_Borough|              DO_Zone|
+-----+----------+-----------------+-----+----------+---------------------+
|  161| Manhattan|   Midtown Center|  141| Manhattan|      Lenox Hill West|
|   43| Manhattan|     Central Park|  237| Manhattan|Upper East Side South|
|   48| Manhattan|     Clinton East|  238| Manhattan|Upper West Side North|
|  138|    Queens|LaGuardia Airport|    7|    Queens|              Astoria|
|  

**Join validation:** The lookup contains **265** records with unique, non-missing location identifiers. Both joins preserve all **37,192,160** eligible trip records. There are **0** unmatched pickup keys and **0** unmatched drop-off keys.

## 2. Reproducible Train/Stream Split

### 2.1 Split Strategy and Validation

The split uses all 37,192,160 eligible, enriched trip records from
`prepared_trips_df`. The training pool receives approximately 75% of these
records for model fitting and evaluation in Part A. The remaining 25% is
reserved for streaming inference in Part B.

These proportions satisfy the assessment requirements of at least 70%
for training and evaluation and at least 20% for streaming. The margin
allows for the small differences between target proportions and actual
record counts produced by `randomSplit()`.

The random seed is fixed at 42. Reproducing the split also requires
consistent source data and Spark partition layout. After the subsets
are saved, the Kafka producer will read the persisted streaming subset.

Streaming records remain excluded from training data exploration,
feature selection, preprocessing fitting, model training and model
selection. The training pool will be divided further into fitting and
evaluation subsets.

Before splitting, the prepared DataFrame is materialised using
`StorageLevel.DISK_ONLY`. This allows successive operations to reuse
the prepared records while limiting memory usage in the environment
with a 2 GB Spark driver. The initial count populates this temporary cache.

In [13]:
from pyspark import StorageLevel

SPLIT_SEED = 42
SPLIT_WEIGHTS = [0.75, 0.25]

# Materialise the prepared records for reuse during split validation
# and the subsequent Parquet writes.
split_source_df = prepared_trips_df.persist(StorageLevel.DISK_ONLY)
split_source_count = split_source_df.count()

assert split_source_count == eligible_trip_count, (
    "The split source count differs from the validated eligible count."
)

# Part A receives the training/evaluation pool.
# Part B receives the separate streaming holdout.
train_df, stream_df = split_source_df.randomSplit(
    SPLIT_WEIGHTS,
    seed=SPLIT_SEED,
)

train_record_count = train_df.count()
stream_record_count = stream_df.count()

train_fraction = train_record_count / split_source_count
stream_fraction = stream_record_count / split_source_count

assert train_record_count + stream_record_count == split_source_count, (
    "The subset counts do not reconcile with the split source."
)

assert train_fraction >= 0.70, (
    "The training/evaluation pool contains less than 70% of eligible records."
)

assert stream_fraction >= 0.20, (
    "The streaming holdout contains less than 20% of eligible records."
)

print("Random seed:", SPLIT_SEED)
print("Target weights:", SPLIT_WEIGHTS)
print("Temporary storage level:", split_source_df.storageLevel)
print(f"Eligible records split: {split_source_count:,}")

Random seed: 42
Target weights: [0.75, 0.25]
Temporary storage level: Disk Serialized 1x Replicated
Eligible records split: 37,192,160


#### Split Audit and Interpretation

The following table compares the target proportions with the measured
subset sizes. `pct_eligible` uses the eligible dataset as its denominator.
`pct_raw` additionally reports each subset as a percentage of the
38,310,226 original source records.

The assertions check the actual proportions before rounding and confirm
that the two subset counts reconcile with the eligible record count.
Only counts are inspected here; streaming feature distributions and
labels are not used to guide modelling decisions.

In [14]:
from IPython.display import Markdown, display

split_audit_rows = [
    (
        "Training/evaluation pool",
        train_record_count,
        100.0 * SPLIT_WEIGHTS[0],
        round(100.0 * train_fraction, 4),
        round(100.0 * train_record_count / raw_trip_count, 4),
    ),
    (
        "Streaming holdout",
        stream_record_count,
        100.0 * SPLIT_WEIGHTS[1],
        round(100.0 * stream_fraction, 4),
        round(100.0 * stream_record_count / raw_trip_count, 4),
    ),
]

spark.createDataFrame(
    split_audit_rows,
    "subset string, record_count long, target_pct double, "
    "pct_eligible double, pct_raw double",
).show(truncate=False)

display(Markdown(
    f"**Split validation:** With seed **{SPLIT_SEED}**, the training "
    f"and evaluation pool contains **{train_record_count:,}** records "
    f"(**{100.0 * train_fraction:.4f}%** of eligible records). "
    f"The streaming holdout contains **{stream_record_count:,}** records "
    f"(**{100.0 * stream_fraction:.4f}%**). "
    f"The counts reconcile with all **{split_source_count:,}** eligible "
    f"records, and both required minimum proportions are satisfied."
))

+------------------------+------------+----------+------------+-------+
|subset                  |record_count|target_pct|pct_eligible|pct_raw|
+------------------------+------------+----------+------------+-------+
|Training/evaluation pool|27895816    |75.0      |75.0046     |72.8156|
|Streaming holdout       |9296344     |25.0      |24.9954     |24.266 |
+------------------------+------------+----------+------------+-------+



**Split validation:** With seed **42**, the training and evaluation pool contains **27,895,816** records (**75.0046%** of eligible records). The streaming holdout contains **9,296,344** records (**24.9954%**). The counts reconcile with all **37,192,160** eligible records, and both required minimum proportions are satisfied.

### 2.2 Subset Persistence and Reload Verification

Both subsets are saved beneath the repository's `data/` directory:

| Saved location | Purpose |
|---|---|
| `data/train_data.parquet` | Training and evaluation pool for Part A |
| `data/stream_data.parquet` | Held-out records replayed through Kafka in Part B |
| `data/split_metadata.json` | Split settings, measured counts and schema |

Saving the training pool also allows Part A to resume without repeating
source preparation and the initial split.

Spark writes each Parquet dataset as a directory containing multiple
part files. The complete directory represents the dataset. Snappy
compression is used, and overwrite mode replaces the generated subsets
when this preparation stage is rerun.

After writing, both datasets are reloaded and checked against their
original counts, column names, column order and data types.

In [15]:
TRAIN_DATA_PATH = DATA_DIR / "train_data.parquet"
STREAM_DATA_PATH = DATA_DIR / "stream_data.parquet"
SPLIT_METADATA_PATH = DATA_DIR / "split_metadata.json"

train_df.write.mode("overwrite").option(
    "compression", "snappy"
).parquet(str(TRAIN_DATA_PATH))

print(
    "Saved training pool:",
    TRAIN_DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
)

stream_df.write.mode("overwrite").option(
    "compression", "snappy"
).parquet(str(STREAM_DATA_PATH))

print(
    "Saved streaming holdout:",
    STREAM_DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
)

Saved training pool: data/train_data.parquet
Saved streaming holdout: data/stream_data.parquet


#### Reload Validation

Each saved dataset must retain its original record count and its column
names, order and types. Nullability flags are excluded from schema
comparison because Spark marks columns nullable when reading Parquet.

A small sample is displayed from the training pool to demonstrate that
the saved data is readable. The streaming subset is checked through its
count and schema only.

Subsequent Part A work will use `train_pool_df`, which is loaded from
the saved training dataset.

In [16]:
train_pool_df = spark.read.parquet(str(TRAIN_DATA_PATH))
stream_reloaded_df = spark.read.parquet(str(STREAM_DATA_PATH))

train_saved_count = train_pool_df.count()
stream_saved_count = stream_reloaded_df.count()


def schema_signature(dataframe):
    """Compare column names, order and types without nullability flags."""
    return [
        (field.name, field.dataType.simpleString())
        for field in dataframe.schema.fields
    ]


expected_schema = schema_signature(split_source_df)

train_schema_matches = (
    schema_signature(train_pool_df) == expected_schema
)
stream_schema_matches = (
    schema_signature(stream_reloaded_df) == expected_schema
)

assert train_saved_count == train_record_count, (
    f"Training count mismatch: expected {train_record_count:,}, "
    f"reloaded {train_saved_count:,}."
)

assert stream_saved_count == stream_record_count, (
    f"Streaming count mismatch: expected {stream_record_count:,}, "
    f"reloaded {stream_saved_count:,}."
)

assert train_schema_matches, "The saved training schema has changed."
assert stream_schema_matches, "The saved streaming schema has changed."

persistence_audit_rows = [
    (
        "Training/evaluation pool",
        train_record_count,
        train_saved_count,
        train_schema_matches,
    ),
    (
        "Streaming holdout",
        stream_record_count,
        stream_saved_count,
        stream_schema_matches,
    ),
]

spark.createDataFrame(
    persistence_audit_rows,
    "subset string, expected_count long, "
    "reloaded_count long, schema_matches boolean",
).show(truncate=False)

print("Verified training and streaming column names and types:")
train_pool_df.printSchema()

train_pool_df.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "pickup_borough",
    "dropoff_borough",
).show(5, truncate=False)

+------------------------+--------------+--------------+--------------+
|subset                  |expected_count|reloaded_count|schema_matches|
+------------------------+--------------+--------------+--------------+
|Training/evaluation pool|27895816      |27895816      |true          |
|Streaming holdout       |9296344       |9296344       |true          |
+------------------------+--------------+--------------+--------------+

Verified training and streaming column names and types:
root
 |-- DOLocationID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |--

#### Split Metadata and Cache Release

A small JSON file records the source filenames, Spark version, session
timezone, random seed, split weights, measured record counts, relative
output paths and reloaded schema. This file can be committed as evidence
of the split configuration.

Once both saved datasets pass validation, the temporary preparation
cache is released. Part A will read the persisted training pool, while
the Part B producer will read `data/stream_data.parquet`.

In [17]:
import json

split_metadata = {
    "dataset": "NYC TLC Yellow Taxi, January–December 2023",
    "source_files": [path.name for path in MONTHLY_FILES],
    "zone_lookup_file": ZONE_FILE.name,
    "spark_version": spark.version,
    "session_timezone": spark.conf.get("spark.sql.session.timeZone"),
    "seed": SPLIT_SEED,
    "weights": {
        "training_pool": SPLIT_WEIGHTS[0],
        "streaming_holdout": SPLIT_WEIGHTS[1],
    },
    "raw_record_count": raw_trip_count,
    "eligible_record_count": split_source_count,
    "training_pool": {
        "path": TRAIN_DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "record_count": train_saved_count,
    },
    "streaming_holdout": {
        "path": STREAM_DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "record_count": stream_saved_count,
    },
    "parquet_schema": train_pool_df.schema.jsonValue(),
}

SPLIT_METADATA_PATH.write_text(
    json.dumps(split_metadata, indent=2) + "\n",
    encoding="utf-8",
)

# Release the temporary cache after successful persistence and validation.
split_source_df.unpersist(blocking=True)

print(
    "Saved split metadata:",
    SPLIT_METADATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
)
print("Temporary preparation cache released.")

display(Markdown(
    f"**Persistence verified:** The saved training pool contains "
    f"**{train_saved_count:,}** records, and the saved streaming "
    f"holdout contains **{stream_saved_count:,}** records. "
    f"Both counts and column signatures match their original subsets. "
    f"Split metadata has been saved, and the temporary cache released."
))

Saved split metadata: data/split_metadata.json
Temporary preparation cache released.


**Persistence verified:** The saved training pool contains **27,895,816** records, and the saved streaming holdout contains **9,296,344** records. Both counts and column signatures match their original subsets. Split metadata has been saved, and the temporary cache released.